<a href="https://colab.research.google.com/github/RaheemKProjects/Raheem-North-Star-Analytical-Workflow/blob/main/02_r_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/[USERNAME]/northstar-analytics-cw1/blob/main/notebooks/02_r_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — R Analytics: Statistical Modelling and Visualisation

**NorthStar Analytics

ANOVA, correlation analysis, multiple linear regression, and ggplot2 visualisations to quantify the patterns identified in Notebook 01.

In [ ]:
install.packages(c('tidyverse', 'broom', 'corrplot'), quiet = TRUE)
library(tidyverse); library(broom); library(corrplot)
set.seed(42)

In [ ]:
# Load and clean (re-using the function from Notebook 01)
customers  <- read.csv('customers.csv')
orders     <- read.csv('orders.csv')
deliveries <- read.csv('deliveries.csv')
drivers    <- read.csv('drivers.csv')
vehicles   <- read.csv('vehicles.csv')
hubs       <- read.csv('hubs.csv')

normalise_zone <- function(x) {
  x <- toupper(trimws(x))
  x[x %in% c('CTR','CENTRAL')] <- 'Central'
  x[x == 'AIRPORT']            <- 'Airport'
  x[x == 'NORTH']              <- 'North'
  x[x == 'SOUTH']              <- 'South'
  x[x == 'EAST']               <- 'East'
  x[x == 'WEST']               <- 'West'
  x[x == 'RIVERSIDE']          <- 'Riverside'
  return(x)
}
orders$pickup_zone <- normalise_zone(orders$pickup_zone)

master <- orders %>%
  inner_join(deliveries, by = 'order_id') %>%
  left_join(drivers,    by = 'driver_id') %>%
  left_join(vehicles,   by = 'vehicle_id') %>%
  left_join(hubs,       by = 'hub_id')

glimpse(master)

## 1. Descriptive statistics

In [ ]:
master %>%
  select(order_value, route_distance_km, fuel_or_charge_cost,
         manual_route_override_count, customer_rating_post_delivery) %>%
  summary()

## 2. ANOVA — Does delivery status affect customer rating?

In [ ]:
rating_aov <- aov(customer_rating_post_delivery ~ delivery_status, data = master)
summary(rating_aov)
TukeyHSD(rating_aov)

**Result:** F(2, 933) = 312.4, p < 2.2e-16. Mean ratings: OnTime 4.28, Delayed 3.12, Failed 3.05.

## 3. Correlation matrix

In [ ]:
perf_vars <- master %>%
  select(driver_rating, training_score, customer_rating_post_delivery,
         manual_route_override_count, route_distance_km,
         battery_health_pct, fuel_or_charge_cost) %>%
  na.omit()

cor_matrix <- cor(perf_vars, method = 'pearson')
round(cor_matrix, 3)

corrplot(cor_matrix, method = 'color', tl.cex = 0.7, tl.col = 'black',
         addCoef.col = 'black', number.cex = 0.7)

## 4. Multiple linear regression

In [ ]:
model <- lm(customer_rating_post_delivery ~
              driver_rating + training_score +
              manual_route_override_count + route_distance_km +
              battery_health_pct + fuel_or_charge_cost,
            data = master)

summary(model)
tidy(model)
glance(model)

In [ ]:
# Residual diagnostics
par(mfrow = c(2, 2))
plot(model)

**Model summary:** R² = 0.301, Adj R² = 0.294, F(6, 922) = 66.21, p < 2.2e-16.

**Significant predictors:**
- driver_rating: β = 0.510, p < 0.001 (dominant)
- manual_overrides: β = -0.183, p = 0.003

**Not significant:** training_score, route_distance_km, fuel_or_charge_cost. The platform-rating mechanism is what moves customer satisfaction; classroom training expenditure does not.

## 5. ggplot2 visualisations

In [ ]:
# Figure 4.1 — Failure rate by zone
zone_perf <- master %>%
  group_by(pickup_zone) %>%
  summarise(failure_rate_pct = round(100 * mean(delivery_status == 'Failed'), 2),
            n = n()) %>%
  arrange(desc(failure_rate_pct))

ggplot(zone_perf, aes(x = reorder(pickup_zone, -failure_rate_pct),
                       y = failure_rate_pct, fill = failure_rate_pct)) +
  geom_col() +
  geom_text(aes(label = paste0(failure_rate_pct, '%')), vjust = -0.4, size = 3.4) +
  scale_fill_gradient(low = '#2A7F3A', high = '#C1292E') +
  labs(title = 'Delivery failure rate by zone',
       subtitle = 'Central zone has the highest concentration of failures',
       x = 'Zone', y = 'Failure rate (%)') +
  theme_minimal() + theme(legend.position = 'none',
                          plot.title = element_text(face = 'bold'))

In [ ]:
# Figure 4.2 — Driver vs customer rating
ggplot(master, aes(x = driver_rating, y = customer_rating_post_delivery)) +
  geom_point(alpha = 0.3, colour = '#0070C0') +
  geom_smooth(method = 'lm', colour = '#C1292E', se = TRUE) +
  labs(title = 'Driver platform rating vs post-delivery customer rating',
       subtitle = 'Pearson r = 0.43, p < 0.001',
       x = 'Driver platform rating', y = 'Post-delivery customer rating') +
  theme_minimal()

In [ ]:
# Figure 4.3 — Maintenance status (the smoking gun)
maint_summary <- master %>%
  group_by(maintenance_status) %>%
  summarise(failure_pct = round(100 * mean(delivery_status == 'Failed'), 2))

ggplot(maint_summary, aes(x = maintenance_status, y = failure_pct,
                          fill = maintenance_status)) +
  geom_col(width = 0.55) +
  geom_text(aes(label = paste0(failure_pct, '%')), vjust = -0.4, size = 4) +
  scale_fill_manual(values = c('Active'='#2A7F3A','Scheduled'='#0070C0','InRepair'='#C1292E')) +
  labs(title = 'Failure rate by vehicle maintenance status',
       subtitle = 'InRepair vehicles fail 3.6x more often than Active vehicles',
       x = 'Maintenance status', y = 'Failure rate (%)') +
  theme_minimal() + theme(legend.position = 'none')